# MedGemma 1.5 4B: smoke test and resolution gate

Two questions, answered before any GPU quota is committed to a sweep:

1. Does the model run on free Kaggle hardware, and how fast?
2. **Does the 896x896 encoding destroy the small print on a full-page A4 report?**

Question 2 decides the shape of the project. MedGemma encodes every image at
896x896 into 256 tokens. A 1588x2246 render is downsampled hard before the
model sees anything, so a `scale` degradation axis could end up measuring a
resize function rather than the model. Four conditions on the same image tell
us which:

| Condition | What it isolates |
|---|---|
| `squash` | The naive baseline |
| `pan_and_scan` | Does tiling recover the small text? |
| `crop_table` | Is the limit resolution, or the model itself? |
| `int4` | Does quantisation cost accuracy on top of that? |

**Setup:** Accelerator `GPU T4 x2`. Attach the dataset of generated reports.
Add `HF_TOKEN` under Add-ons > Secrets.

In [ ]:
!pip install -q -U "transformers>=4.50" accelerate bitsandbytes

In [ ]:
import os, json, time, glob, gc
import torch
from PIL import Image

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print(torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability(0)
print("compute capability:", cap)
# bf16 needs capability >= 8.0. T4 is 7.5 and P100 is 6.0, so both fall back
# to fp16. Google's sample code uses bf16 and will crash or crawl here.
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
print("using dtype:", DTYPE)

In [ ]:
# Recursive, because how deep Kaggle extracts depends on how the zip was built.
# If nothing is found, the tree below distinguishes "dataset not attached" from
# "attached but nested somewhere unexpected", which a bare glob cannot.
ROOT = "/kaggle/input"

if not os.path.isdir(ROOT) or not os.listdir(ROOT):
    raise SystemExit(
        "Nothing under /kaggle/input, so no dataset is attached.\n"
        "Right-hand panel > Input > Add Input > Datasets > add yours.\n"
        "A dataset still processing after upload also shows up empty here."
    )

print("attached under /kaggle/input:")
for dirpath, dirnames, filenames in os.walk(ROOT):
    depth = dirpath.count(os.sep) - ROOT.count(os.sep)
    print("  " * depth + os.path.basename(dirpath) + "/")
    for f in sorted(filenames)[:4]:
        print("  " * (depth + 1) + f)
    if len(filenames) > 4:
        print("  " * (depth + 1) + f"... {len(filenames) - 4} more")

DATA = sorted(glob.glob(f"{ROOT}/**/*.png", recursive=True))
print(f"\n{len(DATA)} images found")
if not DATA:
    raise SystemExit(
        "Dataset is attached but contains no PNGs. Check that the upload held\n"
        "the images themselves, and that Kaggle finished unzipping it."
    )

IMG_PATH = DATA[0]
GT_PATH = IMG_PATH.replace(".png", ".json")
if not os.path.exists(GT_PATH):
    raise SystemExit(f"Found {IMG_PATH} but no ground truth beside it at {GT_PATH}")

gt = json.load(open(GT_PATH))
img = Image.open(IMG_PATH).convert("RGB")
print("using:", IMG_PATH)
print("image size:", img.size)
print("ground truth tests:", len(gt["tests"]))

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"

t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,          # 'dtype', not the deprecated 'torch_dtype'
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
load_s = time.time() - t0
print(f"loaded in {load_s:.1f}s")
print(f"weights VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
PROMPT = """Extract the laboratory test results from this report.

Return ONLY a JSON object, no explanation and no markdown fences:

{"patient": {"name": "", "age": null, "sex": ""},
 "tests": [{"test_name": "", "value": null, "unit": "",
            "ref_low": null, "ref_high": null, "ref_text": "", "flag": ""}]}

Rules:
- Copy test names exactly as printed.
- One entry per row of the results table.
- flag is "H", "L" or "" exactly as printed.
- Do not invent any value that is not printed on the report.
"""


def extract(image, pan_and_scan=False, max_new_tokens=800):
    """Run one extraction. Returns a dict of everything worth recording.

    do_pan_and_scan is passed to apply_chat_template, which routes it into the
    processor's images_kwargs. Setting it on processor.image_processor does
    nothing: the defaults in Gemma3ProcessorKwargs are re-applied on every call
    and silently overwrite the attribute, so the run looks like it tiled when
    it did not. input_tokens below is the check that it actually took effect.
    """
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
        do_pan_and_scan=pan_and_scan,
    ).to(model.device, dtype=DTYPE)

    input_len = inputs["input_ids"].shape[-1]

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    secs = time.time() - t0

    new_tokens = out.shape[-1] - input_len
    text = processor.decode(out[0][input_len:], skip_special_tokens=True)
    return dict(
        text=text,
        seconds=round(secs, 1),
        peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
        input_tokens=input_len,
        output_tokens=int(new_tokens),
        # If this is True the model never emitted a stop token and ran to the
        # cap, which is why a run takes 100s instead of 25s.
        hit_token_cap=bool(new_tokens >= max_new_tokens),
    )

In [ ]:
import re


def parse_json(text):
    """Pull the first complete JSON object out of the model's output.

    The naive first-brace-to-last-brace slice fails on real output two ways:
    trailing prose after the object gives 'Extra data', and a second object
    swallows everything between them. Walking to the matching brace, while
    respecting string literals and escapes, handles both.
    """
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    start = cleaned.find("{")
    if start == -1:
        return None, "no_json_found"

    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(cleaned[start:], start):
        if esc:
            esc = False
        elif ch == "\\":
            esc = True
        elif ch == '"':
            in_str = not in_str
        elif not in_str:
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(cleaned[start:i + 1]), None
                    except json.JSONDecodeError as err:
                        return None, f"parse_error: {err}"
    return None, "unterminated_json"


def score(pred, gt):
    """Crude value recall. Not the real harness, just enough signal to say
    whether the model read the page at all."""
    if not pred or not isinstance(pred.get("tests"), list):
        return 0.0, 0
    want = {round(float(t["value"]), 2) for t in gt["tests"]}
    got = set()
    for t in pred["tests"]:
        v = t.get("value")
        try:
            got.add(round(float(v), 2))
        except (TypeError, ValueError):
            pass
    return len(want & got) / len(want), len(pred["tests"])

In [ ]:
w, h = img.size
img_crop = img.crop((0, 0, w, int(h * 0.45)))   # results table sits in the top ~45%
print("full:", img.size, " crop:", img_crop.size)

results = {}
for name, image, pas in [
    ("squash", img, False),
    ("pan_and_scan", img, True),
    ("crop_table", img_crop, False),
]:
    r = extract(image, pan_and_scan=pas)
    pred, err = parse_json(r["text"])
    recall, n_tests = score(pred, gt)
    r.update(parse_error=err, tests_returned=n_tests, value_recall=round(recall, 3))
    results[name] = r

    print(f"\n{'='*70}\n{name}")
    print(f"  {r['seconds']}s | in={r['input_tokens']} out={r['output_tokens']}"
          f" | cap_hit={r['hit_token_cap']} | tests={n_tests} recall={recall:.2f}"
          f" | {err or 'parsed ok'}")
    print("  --- raw output ---")
    print("  " + r["text"][:1500].replace("\n", "\n  "))

# Verify the tiling actually engaged instead of assuming it. Pan and scan adds
# 256 tokens per extra crop, so an unchanged count means it silently did not run
# and the two conditions are the same experiment twice.
delta = results["pan_and_scan"]["input_tokens"] - results["squash"]["input_tokens"]
print(f"\n{'='*70}")
print(f"pan_and_scan added {delta} input tokens "
      f"({results['squash']['input_tokens']} -> {results['pan_and_scan']['input_tokens']})")
if delta == 0:
    print("  WARNING: pan and scan did not engage. Check that do_pan_and_scan")
    print("  reaches the processor, and that the image aspect ratio exceeds")
    print("  pan_and_scan_min_ratio_to_activate (default 1.2).")
else:
    print(f"  OK: roughly {delta // 256} extra crop(s) were encoded.")

# Save raw outputs. If the numbers look wrong, the text is what explains why.
with open("/kaggle/working/raw_outputs.json", "w") as f:
    json.dump({k: v["text"] for k, v in results.items()}, f, indent=2)

In [ ]:
# 4-bit. Reloads the model, so run only after the fp16 numbers are recorded.
del model
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
print(f"4-bit loaded in {time.time()-t0:.1f}s, "
      f"weights VRAM {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Match whichever fp16 condition read the page best, so the comparison isolates
# quantisation rather than confounding it with the resolution question.
best = max(("squash", "pan_and_scan", "crop_table"),
           key=lambda k: results[k]["value_recall"])
best_image = img_crop if best == "crop_table" else img

r = extract(best_image, pan_and_scan=(best == "pan_and_scan"))
pred, err = parse_json(r["text"])
recall, n_tests = score(pred, gt)
r.update(parse_error=err, tests_returned=n_tests, value_recall=round(recall, 3),
         matched_condition=best)
results["int4"] = r

print(f"\nint4 vs fp16 on '{best}':")
print(f"  fp16 {results[best]['seconds']}s recall={results[best]['value_recall']:.2f}")
print(f"  int4 {r['seconds']}s recall={recall:.2f}")
print("  --- raw output ---")
print("  " + r["text"][:1000].replace("\n", "\n  "))

In [ ]:
summary = {
    "model": MODEL_ID, "gpu": torch.cuda.get_device_name(0), "dtype": str(DTYPE),
    "load_seconds": round(load_s, 1), "image": os.path.basename(IMG_PATH),
    "image_size": list(img.size), "gt_test_count": len(gt["tests"]),
    "conditions": {k: {kk: vv for kk, vv in v.items() if kk != "text"}
                   for k, v in results.items()},
}
with open("/kaggle/working/smoke_test.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"{'condition':<14}{'sec':>7}{'in_tok':>8}{'out_tok':>9}{'cap':>6}"
      f"{'tests':>7}{'recall':>8}  parse")
for k, v in results.items():
    print(f"{k:<14}{v['seconds']:>7}{v['input_tokens']:>8}{v['output_tokens']:>9}"
          f"{str(v['hit_token_cap']):>6}{v['tests_returned']:>7}"
          f"{v['value_recall']:>8.2f}  {v['parse_error'] or 'ok'}")

print(f"\nground truth had {len(gt['tests'])} tests")
print(json.dumps(summary, indent=2))